### Import relevant libraries

In [1]:
import pandas as pd
import numpy as np

### Load dataset

In [2]:
image_encoder_type = 'clip'  # 'clip' or 'vit'
image_encoder_type

'clip'

In [3]:
if image_encoder_type == 'clip':
    # Load the data extracted from video frames using CLIP pretrained model
    df_action = pd.concat([pd.read_csv(f'../datasets/video_frame_features_clip_part_{part}.csv') for part in range(6)], ignore_index=True)
elif image_encoder_type == 'vit':
    # Load the data extracted from video frames using ViT pretrained model
    df_action = pd.concat([pd.read_csv(f'../datasets/video_frame_features_vit_part_{part}.csv') for part in range(7)], ignore_index=True)


In [4]:
df_action.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511,feature_512,video_file_name,label,subset
0,0.000935,-0.3577,0.2145,-0.08440,-0.09460,-0.02750,-0.08655,-0.11280,0.1923,0.03400,...,-0.1499,-0.2192,0.1538,-0.09460,0.4114,-0.1346,0.4624,uccrime_Robbery138_x264_trimmed.mp4,abnormal,train
1,-0.000123,-0.3880,0.2308,-0.08770,-0.11510,-0.04703,-0.07904,-0.07840,0.1633,0.03420,...,-0.1407,-0.2595,0.1459,-0.10730,0.3780,-0.1385,0.4746,uccrime_Robbery138_x264_trimmed.mp4,abnormal,train
2,-0.004620,-0.3840,0.2339,-0.09375,-0.11914,-0.04355,-0.07587,-0.06775,0.1633,0.02902,...,-0.1410,-0.2610,0.1477,-0.10970,0.3623,-0.1371,0.4739,uccrime_Robbery138_x264_trimmed.mp4,abnormal,train
3,-0.002085,-0.3810,0.2395,-0.09644,-0.12110,-0.04382,-0.07470,-0.06055,0.1484,0.02753,...,-0.1399,-0.2583,0.1461,-0.11316,0.3594,-0.1392,0.4707,uccrime_Robbery138_x264_trimmed.mp4,abnormal,train
4,-0.000967,-0.3745,0.2433,-0.09955,-0.12006,-0.04300,-0.06650,-0.04440,0.1410,0.01003,...,-0.1443,-0.2595,0.1571,-0.11110,0.3333,-0.1255,0.4797,uccrime_Robbery138_x264_trimmed.mp4,abnormal,train


In [5]:
df_action.tail()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511,feature_512,video_file_name,label,subset
1576717,0.3200,0.2773,0.5000,-0.5870,0.2595,-0.3477,0.07916,0.004738,-0.18290,-0.01791,...,-0.2530,-0.1477,-0.10516,-0.21020,0.7764,-0.06080,0.04840,uccrime_Stealing017_x264_trimmed_augmented.mp4,abnormal,train
1576718,0.2351,0.4080,0.4778,-0.4307,0.2489,-0.3242,0.18820,0.154700,0.18920,0.06560,...,-0.3180,-0.1761,-0.13840,-0.12286,0.8276,-0.14610,0.12520,uccrime_Stealing017_x264_trimmed_augmented.mp4,abnormal,train
1576719,0.2952,0.3225,0.4514,-0.5767,0.2583,-0.3064,0.10450,0.016950,-0.06155,0.02588,...,-0.2318,-0.1964,-0.03760,-0.15280,0.8315,-0.09875,0.02907,uccrime_Stealing017_x264_trimmed_augmented.mp4,abnormal,train
1576720,0.1976,0.4204,0.4653,-0.3586,0.2832,-0.3472,0.26370,0.219000,0.24710,0.03650,...,-0.3828,-0.1595,-0.14050,-0.16930,0.8830,-0.11000,0.10974,uccrime_Stealing017_x264_trimmed_augmented.mp4,abnormal,train
1576721,0.2502,0.4526,0.5073,-0.3860,0.3120,-0.2832,0.26460,0.285600,0.28960,-0.01590,...,-0.3743,-0.1514,-0.13760,-0.13090,0.7970,-0.12366,0.12920,uccrime_Stealing017_x264_trimmed_augmented.mp4,abnormal,train


In [6]:
df_action.shape

(1576722, 515)

### Explore Data

In [7]:
# Check for missing values
df_action.isnull().sum()

feature_1          0
feature_2          0
feature_3          0
feature_4          0
feature_5          0
                  ..
feature_511        0
feature_512        0
video_file_name    0
label              0
subset             0
Length: 515, dtype: int64

In [8]:
# Check for label distribution
df_action['label'].value_counts()

label
abnormal    1227047
normal       349675
Name: count, dtype: int64

In [9]:
# Check number of unique video files per label
df_action.groupby('label')['video_file_name'].nunique()

label
abnormal    1157
normal      1149
Name: video_file_name, dtype: int64

In [10]:
# Check number of unieque video files
df_action['video_file_name'].nunique()

2306

In [11]:
# Check number of labels by subset
df_action.groupby('subset')['label'].value_counts()

subset  label   
test    abnormal    217996
        normal       52197
train   abnormal    822152
        normal      245337
val     abnormal    186899
        normal       52141
Name: count, dtype: int64

In [12]:
# Check number of unique video files per subset
df_action.groupby('subset')['video_file_name'].nunique()

subset
test      352
train    1603
val       351
Name: video_file_name, dtype: int64

In [13]:
df_action['subset'].unique()

array(['train', 'val', 'test'], dtype=object)

### Data preparation

In [14]:
SEED = 124
np.random.seed(SEED)

In [15]:
# Helper function to convert series to supervised learning
def series_to_supervised(data, column_names, n_in=1, n_out=1, dropnan=True, n_out_index=None, is_drop_target=False, lag_sampling=1, sliding_stride=1):
    """
    convert series to supervised learning
    Params:
    - n_in: number of timesteps
    - n_out: number of output timestep for prediction | eg. 1 day, 2 days, ....
    - n_out_index: define specific variable for prediction
    - is_drop_target: drop target variable from input variables (to be used for classification problem)
    - lag_sampling: lag sampling stride (to skip some samples within a window)
    - sliding_stride: sliding window stride (to skip some windows)
    - dropnan: boolean whether or not to drop rows with NaN values
    Returns:
    - Pandas DataFrame of series framed for supervised learning.
    """
    df_new = pd.DataFrame(data)
    
    if is_drop_target:
        df_input = df_new.drop(columns=[column_names[n_out_index]])
    else:
        df_input = df_new.copy()
    
    n_vars = df_input.shape[1]
    
    cols, names = list(), list()
    
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -lag_sampling):
        cols.append(df_input.shift(i))
        names += [f'{column_names[j]}(t-{i})' for j in range(n_vars)]
    
    # forecast sequence (t, t+1, ..., t+n)
    for i in range(0, n_out):
        if n_out_index:
            cols.append(df_new.shift(-i).iloc[:, n_out_index])
            if i == 0:
                names += [f'{column_names[n_out_index]}(t)']
            else:
                names += [f'{column_names[n_out_index]}(t+{i})']
        else:
            cols.append(df_new.shift(-i))
            if i == 0:
                names += [f'{column_names[j]}(t)' for j in range(n_vars)]
            else:
                names += [f'{column_names[j]}(t+{i})' for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values that generated from df_new.shift
    if dropnan:
        agg.dropna(inplace=True)

    # Use slicing to select only the rows corresponding to the desired stride
    if sliding_stride > 1:
        agg = agg.iloc[::sliding_stride, :]
    
    return agg

In [16]:
# define features
column_names = df_action.columns.tolist()
column_names.remove('video_file_name')
column_names.remove('subset')
column_names

['feature_1',
 'feature_2',
 'feature_3',
 'feature_4',
 'feature_5',
 'feature_6',
 'feature_7',
 'feature_8',
 'feature_9',
 'feature_10',
 'feature_11',
 'feature_12',
 'feature_13',
 'feature_14',
 'feature_15',
 'feature_16',
 'feature_17',
 'feature_18',
 'feature_19',
 'feature_20',
 'feature_21',
 'feature_22',
 'feature_23',
 'feature_24',
 'feature_25',
 'feature_26',
 'feature_27',
 'feature_28',
 'feature_29',
 'feature_30',
 'feature_31',
 'feature_32',
 'feature_33',
 'feature_34',
 'feature_35',
 'feature_36',
 'feature_37',
 'feature_38',
 'feature_39',
 'feature_40',
 'feature_41',
 'feature_42',
 'feature_43',
 'feature_44',
 'feature_45',
 'feature_46',
 'feature_47',
 'feature_48',
 'feature_49',
 'feature_50',
 'feature_51',
 'feature_52',
 'feature_53',
 'feature_54',
 'feature_55',
 'feature_56',
 'feature_57',
 'feature_58',
 'feature_59',
 'feature_60',
 'feature_61',
 'feature_62',
 'feature_63',
 'feature_64',
 'feature_65',
 'feature_66',
 'feature_67',
 'fe

In [26]:
for i in range(10, 0, -5):
    print(i)

10
5


In [39]:
# Define window_size (number of timesteps or number of frames), number of features, and number of output step
window_size = 120 # 120 frames ~ 4 seconds
lag_sampling = 4 # step size for selecting frames within a window
sliding_stride = 3 # sliding window stride
num_features = len(column_names[:-1])
n_steps_out = 1
n_out_index =  -1

In [40]:
# Reframe as supervised learning by each video file
subsets = df_action['subset'].unique()
df_by_subset = {}

for subset in subsets:
    print(f'Processing subset: {subset}')
    df_action_subset = df_action[df_action['subset'] == subset]
    video_files = df_action_subset['video_file_name'].unique()

    dfs = []
    n_skip = 0
    for i, video_file in enumerate(video_files):
        print(f'Processing video file {i+1}/{len(video_files)}: {video_file}'.ljust(200), end='\r')
        df_video = df_action_subset[df_action_subset['video_file_name'] == video_file]
        if df_video.shape[0] < int(window_size/lag_sampling):
            n_skip += 1
            continue
        elif df_video.shape[0] < window_size:
            idx = np.linspace(0, df_video.shape[0]-1, num=int(window_size/lag_sampling), dtype=int)
            df_video = df_video.iloc[idx]

        df_reframed = series_to_supervised(
            df_video.drop(columns=['video_file_name', 'subset']),
            column_names,
            n_in=window_size,
            n_out=n_steps_out,
            n_out_index=n_out_index,
            is_drop_target=True,
            lag_sampling=lag_sampling,
            sliding_stride=sliding_stride
        )
        dfs.append(df_reframed)

    df_subset = pd.concat(dfs, ignore_index=True)
    df_by_subset[subset] = df_subset
    print(f"Finished processing subset: {subset}. Skipped {n_skip} videos.".ljust(200))


Processing subset: train
Finished processing subset: train. Skipped 10 videos.                                                                                                                                                   
Processing subset: val
Finished processing subset: val. Skipped 2 videos.                                                                                                                                                      
Processing subset: test
Finished processing subset: test. Skipped 1 videos.                                                                                                                                                     


In [41]:
df_by_subset['train'].head()

,feature_1(t-120),feature_2(t-120),feature_3(t-120),feature_4(t-120),feature_5(t-120),feature_6(t-120),feature_7(t-120),feature_8(t-120),feature_9(t-120),feature_10(t-120),...,feature_504(t-4),feature_505(t-4),feature_506(t-4),feature_507(t-4),feature_508(t-4),feature_509(t-4),feature_510(t-4),feature_511(t-4),feature_512(t-4),label(t)
0,0.000935,-0.3577,0.2145,-0.08440,-0.0946,-0.027500,-0.08655,-0.112800,0.1923,0.03400,...,-0.4639,-0.11190,-0.3306,-0.3467,0.06085,-0.015630,0.1912,-0.030410,0.5330,abnormal
1,-0.002085,-0.3810,0.2395,-0.09644,-0.1211,-0.043820,-0.07470,-0.060550,0.1484,0.02753,...,-0.4722,-0.06964,-0.2367,-0.2175,0.13110,-0.001121,0.3690,-0.058300,0.3950,abnormal
2,0.027910,-0.3550,0.2617,-0.08997,-0.1370,-0.018310,-0.07850,-0.045400,0.1298,-0.02762,...,-0.5520,-0.08685,-0.1158,-0.1686,0.12830,0.078500,0.4778,-0.151700,0.3362,abnormal
3,0.051240,-0.3796,0.2485,-0.10730,-0.1622,-0.008484,-0.04718,-0.003565,0.1837,0.05255,...,-0.4387,-0.01785,-0.1619,-0.0867,0.22810,-0.086900,0.5030,0.002449,0.2537,abnormal
4,0.067900,-0.3735,0.2448,-0.11957,-0.1400,0.002785,-0.03920,-0.015010,0.1088,-0.01381,...,-0.4390,0.00516,-0.0994,-0.1694,0.26300,-0.067800,0.4854,0.010120,0.2260,abnormal


In [42]:
# Check dataframes size for each subset
for key in df_by_subset.keys():
    print(f'Subset: {key}, Shape: {df_by_subset[key].shape}')

Subset: train, Shape: (297718, 15361)
Subset: val, Shape: (66983, 15361)
Subset: test, Shape: (77480, 15361)


In [43]:
# Rename label column
for key in df_by_subset.keys():
    df_by_subset[key].rename(columns={f'label(t)': 'label'}, inplace=True)

In [44]:
# Check label distribution for each subset
for key in df_by_subset.keys():
    print(f'Subset: {key} \n{df_by_subset[key]["label"].value_counts()}')

Subset: train 
label
abnormal    243680
normal       54038
Name: count, dtype: int64
Subset: val 
label
abnormal    55671
normal      11312
Name: count, dtype: int64
Subset: test 
label
abnormal    66083
normal      11397
Name: count, dtype: int64


In [45]:
# Define sample size for each subset
# train: 80%, val: 10%, test: 10%
sample_size = {
    'train': 80000,
    'val': 10000,
    'test': 10000
}

In [46]:
# Sample data for each subset
df_train = pd.concat([df_by_subset['train'][df_by_subset['train']['label'] == label].sample(n=int(sample_size['train']/2), random_state=SEED) for label in df_by_subset['train']['label'].unique()])
df_train.shape

(80000, 15361)

In [47]:
df_val = pd.concat([df_by_subset['val'][df_by_subset['val']['label'] == label].sample(n=int(sample_size['val']/2), random_state=SEED) for label in df_by_subset['val']['label'].unique()])
df_val.shape

(10000, 15361)

In [48]:
df_test = pd.concat([df_by_subset['test'][df_by_subset['test']['label'] == label].sample(n=int(sample_size['test']/2), random_state=SEED) for label in df_by_subset['test']['label'].unique()])
df_test.shape

(10000, 15361)

In [49]:
# Check label distribution after sampling
print(f'Train label distribution:\n{df_train["label"].value_counts()}') 
print(f'Validation label distribution:\n{df_val["label"].value_counts()}')
print(f'Test label distribution:\n{df_test["label"].value_counts()}')

Train label distribution:
label
abnormal    40000
normal      40000
Name: count, dtype: int64
Validation label distribution:
label
abnormal    5000
normal      5000
Name: count, dtype: int64
Test label distribution:
label
abnormal    5000
normal      5000
Name: count, dtype: int64


In [50]:
# Convert labels to binary (0: normal, 1: abnormal)
df_train['label'] = df_train['label'].map(lambda x: 0 if x == 'abnormal' else 1)
df_val['label'] = df_val['label'].map(lambda x: 0 if x == 'abnormal' else 1)
df_test['label'] = df_test['label'].map(lambda x: 0 if x == 'abnormal' else 1)

In [51]:
df_train.head()

,feature_1(t-120),feature_2(t-120),feature_3(t-120),feature_4(t-120),feature_5(t-120),feature_6(t-120),feature_7(t-120),feature_8(t-120),feature_9(t-120),feature_10(t-120),...,feature_504(t-4),feature_505(t-4),feature_506(t-4),feature_507(t-4),feature_508(t-4),feature_509(t-4),feature_510(t-4),feature_511(t-4),feature_512(t-4),label
32259,-0.01907,-0.23840,-0.4560,-0.1912,-0.12300,0.23740,0.30900,-0.1070,-0.6113,-0.11176,...,-0.24540,0.3367,-0.12570,0.450200,0.2004,0.3748,1.2010,0.039580,-0.02371,0
290636,0.25730,0.37350,0.4255,-0.3743,-0.05515,0.01149,0.24170,-0.1268,0.3640,-0.06540,...,-0.02477,0.3635,-0.75300,-0.076660,0.0810,0.1514,0.5930,-0.462600,0.09630,0
153716,0.01997,-0.12244,0.3381,-0.2173,0.13650,0.39650,0.17880,0.0884,-0.1783,0.01465,...,-0.16540,-0.1866,0.14270,-0.046600,0.2778,0.1487,0.7930,0.021240,-0.12225,0
26682,0.15040,-0.01156,0.1420,-0.1382,-0.09800,-0.09160,-0.03085,-0.1042,-0.4185,-0.06590,...,0.13430,-0.1093,-0.12213,0.007347,0.2479,0.2435,0.7227,0.135500,0.03430,0
148389,0.06710,-0.37650,0.3755,-0.2056,0.07230,-0.16410,-0.01796,-0.3179,-0.2556,-0.34080,...,-0.65500,-0.6113,0.17130,0.075560,0.1512,0.1280,0.1328,0.014534,0.25800,0


In [52]:
df_train.tail()

,feature_1(t-120),feature_2(t-120),feature_3(t-120),feature_4(t-120),feature_5(t-120),feature_6(t-120),feature_7(t-120),feature_8(t-120),feature_9(t-120),feature_10(t-120),...,feature_504(t-4),feature_505(t-4),feature_506(t-4),feature_507(t-4),feature_508(t-4),feature_509(t-4),feature_510(t-4),feature_511(t-4),feature_512(t-4),label
193361,0.02834,0.2268,0.11060,-0.182400,0.1072,0.0712,0.3816,0.2595,-0.0479,-0.205100,...,0.334200,0.21020,0.1594,-0.07745,0.000959,0.3810,0.5300,-0.5290,-0.2632,1
172108,-0.08230,0.0971,0.30200,-0.035430,0.2646,-0.2081,0.3599,0.2334,0.2925,-0.324700,...,-0.004044,-0.11554,-0.3130,-0.00842,0.142200,0.5215,0.6504,-0.5730,-0.5470,1
195110,-0.13300,0.3208,-0.01784,-0.251700,0.0959,-0.2786,0.2957,0.6240,0.2769,-0.035500,...,0.031680,0.75700,-0.2844,-0.07880,-0.128300,-0.1434,1.0680,-0.0838,-0.2211,1
162116,-0.05460,0.0510,0.03790,0.001488,-0.0772,-0.1543,0.2888,0.5710,-0.3125,-0.423600,...,0.465600,0.47440,0.3384,-0.02522,-0.270300,0.1860,0.7870,-0.5140,-0.5680,1
205429,-0.21830,0.3600,0.06073,-0.187100,0.5903,-0.2790,0.2388,0.1543,-0.1937,0.000981,...,0.144200,-0.66800,-0.2360,-0.24830,0.053280,0.4028,0.6760,-0.5890,-0.6070,1


In [53]:
# Reshape data for model training
X_train, y_train = df_train.drop(columns=['label']).values, df_train['label'].values
X_val, y_val = df_val.drop(columns=['label']).values, df_val['label'].values
X_test, y_test = df_test.drop(columns=['label']).values, df_test['label'].values

effective_window_size = int(window_size / lag_sampling)
X_train = X_train.reshape((-1, effective_window_size, num_features))
X_val = X_val.reshape((-1, effective_window_size, num_features))
X_test = X_test.reshape((-1, effective_window_size, num_features))

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((80000, 30, 512),
 (80000,),
 (10000, 30, 512),
 (10000,),
 (10000, 30, 512),
 (10000,))

In [ ]:
# Save processed data to csv files
np.savez_compressed(f'../datasets/video_frame_features_{image_encoder_type}_ws{window_size}_ls{lag_sampling}_supervised_data.npz', 
                    X_train=X_train, y_train=y_train, 
                    X_val=X_val, y_val=y_val, 
                    X_test=X_test, y_test=y_test)